# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [4]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [5]:
CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}
import pandas as pd

In [6]:
# 1
def clean(path):
    df = pd.read_csv(path)
    print(len(df))

    df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
    print(len(df))

    df["city"] = df["city"].str.strip().str.lower().map(CITY)
    print(len(df))

    df = df.drop_duplicates().copy()
    print(len(df))

    df["order_date"] = pd.to_datetime(df["order_date"])
    df["month"] = df["order_date"].dt.month
    print(len(df))

    df = df[df["customer_id"] != "TEST"].copy()
    print(len(df))
    print(df["amount"].isna().sum())
    print(df["city"].isna().sum())
    print(df["order_id"].is_unique)
    print(len(df))
    return df

df_clean = clean("orders.csv")

5463
5463
5463
5283
5283
5280
0
0
True
5280


## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [7]:
# 2
paid = df_clean["status"] == "оплачен"
sep = df_clean["month"] == 9
octo = df_clean["month"] == 10

s_all = df_clean.loc[paid & sep, "amount"].sum()
o_all = df_clean.loc[paid & octo, "amount"].sum()
print(s_all, o_all, f"{o_all / s_all - 1:.1%}")

cities = ["Алматы", "Астана", "Шымкент", "Караганда", "Актобе", "Атырау"]
for city in cities:
    c_mask = df_clean["city"] == city
    s_c = df_clean.loc[paid & c_mask & sep, "amount"].sum()
    o_c = df_clean.loc[paid & c_mask & octo, "amount"].sum()
    print(city, s_c, o_c, f"{o_c / s_c - 1:.1%}")

35243410.0 34419520.0 -2.3%
Алматы 14501070.0 14643020.0 1.0%
Астана 9034220.0 8850530.0 -2.0%
Шымкент 4558650.0 4044010.0 -11.3%
Караганда 3648520.0 3371680.0 -7.6%
Актобе 2576960.0 2540420.0 -1.4%
Атырау 923990.0 969860.0 5.0%


## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

In [8]:
# 3
raw = pd.read_csv("orders.csv")
raw["amount"] = pd.to_numeric(raw["amount"].str.replace(" ", ""))
raw["city"] = raw["city"].str.strip().str.lower().map(CITY)

p0 = raw["status"] == "оплачен"
s0 = raw["order_date"] < "2026-10"
o0 = ~s0
print("Шаг 0:", len(raw), f"{raw.loc[p0 & o0, 'amount'].sum() / raw.loc[p0 & s0, 'amount'].sum() - 1:.1%}")

no_dup = raw.drop_duplicates().copy()
p1 = no_dup["status"] == "оплачен"
s1 = no_dup["order_date"] < "2026-10"
o1 = ~s1
print("После drop_duplicates:", len(no_dup), f"{no_dup.loc[p1 & o1, 'amount'].sum() / no_dup.loc[p1 & s1, 'amount'].sum() - 1:.1%}")

no_test = no_dup[no_dup["customer_id"] != "TEST"].copy()
p2 = no_test["status"] == "оплачен"
s2 = no_test["order_date"] < "2026-10"
o2 = ~s2
print("После удаления TEST:", len(no_test), f"{no_test.loc[p2 & o2, 'amount'].sum() / no_test.loc[p2 & s2, 'amount'].sum() - 1:.1%}")

Шаг 0: 5463 -22.7%
После drop_duplicates: 5283 -19.6%
После удаления TEST: 5280 -2.3%


## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [10]:
# 4
print(df_clean["delivery_days"].isna().sum())
print(df_clean.loc[df_clean["delivery_days"].isna(), "city"].value_counts())
print(df_clean.loc[df_clean["delivery_days"].isna(), "status"].value_counts())

d_sh = df_clean.loc[df_clean["city"] == "Шымкент", "delivery_days"]
med_sh = d_sh.median()

print(f"dropna: {(d_sh.dropna() <= 2).mean():.1%}")
print(f"fillna 1: {(d_sh.fillna(med_sh) <= 2).mean():.1%}")
print(f"fillna 2: {(d_sh.fillna(0) <= 2).mean():.1%}")

158
city
Алматы       63
Астана       40
Караганда    24
Шымкент      22
Атырау        5
Актобе        4
Name: count, dtype: int64
status
оплачен    144
отменён     14
Name: count, dtype: int64
dropna: 29.8%
fillna 1: 29.0%
fillna 2: 31.8%


## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [11]:
# 5
df = pd.read_csv("orders.csv")
df = df.drop_duplicates().copy()
df["city"] = df["city"].str.strip().str.lower().map(CITY)
print(df["city"].nunique(), len(df))

6 5283


## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [12]:
# 6
days_cnt = df_clean["order_date"].dt.day_name().value_counts()
print(days_cnt)

min_cnt = days_cnt.min()
max_cnt = days_cnt.max()
print(days_cnt.idxmin(), min_cnt)
print(days_cnt.idxmax(), max_cnt)
print(f"{min_cnt / max_cnt - 1:.1%}")

order_date
Thursday     786
Tuesday      785
Friday       781
Wednesday    774
Saturday     773
Sunday       697
Monday       684
Name: count, dtype: int64
Monday 684
Thursday 786
-13.0%


## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [ ]:
# 7

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [ ]:
# 8